# 🚀 EduFocus AI — Huấn Luyện Toàn Bộ Pipeline Trên RAF-DB (100 Epochs)
### Thực thi quy trình End-to-End trực tiếp từ GitHub trên Google Colab GPU (T4 / V100 / A100)

Lệnh thực thi chính:
```bash
python backend/evaluation/run_complete_pipeline.py --dataset rafdb --epochs 100 --patience 15
```

Quy trình tự động thực hiện 4 giai đoạn khép kín:
1. **Huấn luyện Rã băng tuần tự (Sequential Gradual Unfreezing 100 Epochs, Patience 15)**:
   - Tối ưu hóa phân tầng OpenAI CLIP ViT-B/32 và Learnable Facial Query Cross-Attention trực tiếp trên GPU.
   - Lưu checkpoint tốt nhất tại `backend/models/clip_finetuned_best.pth` & `facial_cross_attn_best.pth`.
2. **Trích xuất đặc trưng 544D với Backbone đã tinh chỉnh**:
   - Sinh file lưu đệm siêu tốc `backend/cache_rafdb.npz` (12,271 train + 3,068 test).
3. **Tối ưu hóa không gian đặc trưng bằng EPSO**:
   - Sử dụng thuật toán bầy đàn tiến hóa lọc 255 chiều nhiễu, giữ lại 289 đặc trưng nhạy cảm nhất.
4. **Huấn luyện Soft-Voting Ensemble Head**:
   - Huấn luyện `EmotionResMLP` + `XGBoost Classifier` + `Baseline MLP`.
   - Đạt độ chính xác SOTA trên RAF-DB test set.
---


## 1️⃣ Kiểm tra Phần cứng GPU (NVIDIA T4 / V100 / A100)


In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise SystemError("⚠️ Vui lòng bật GPU trước khi chạy: Vào menu Runtime -> Change runtime type -> T4 GPU!")


## 2️⃣ Tải Mã Nguồn Trực Tiếp từ GitHub (`git clone`)
Tải mã nguồn mới nhất từ kho lưu trữ GitHub của dự án vào môi trường Google Colab.


In [ ]:
#@title 📥 Clone Mã Nguồn từ GitHub { run: "auto" }
import os
import sys

GITHUB_REPO_URL = "https://github.com/DungLe2005/Emotion_Concentration-.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
PROJECT_DIR = "/content/Emotion_Concentration"

if not os.path.exists(PROJECT_DIR):
    print(f"🚀 Đang clone mã nguồn từ GitHub: {GITHUB_REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}
    print("✅ Đã clone xong mã nguồn!")
else:
    print(f"📁 Thư mục {PROJECT_DIR} đã tồn tại. Đang cập nhật code mới nhất từ git pull...")
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

# Thiết lập biến môi trường và thư mục làm việc
os.environ["PROJECT_ROOT"] = PROJECT_DIR
os.chdir(PROJECT_DIR)

BACKEND_DIR = os.path.join(PROJECT_DIR, "backend")
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f"\n✅ Thư mục dự án: {PROJECT_DIR}")
print(f"✅ Thư mục backend: {BACKEND_DIR}")
print(f"✅ Working Directory: {os.getcwd()}")


## 3️⃣ Cài đặt các Thư viện Cần thiết


In [ ]:
# Cài đặt OpenAI CLIP, MediaPipe, RetinaFace, XGBoost, Scikit-learn
!pip install git+https://github.com/openai/CLIP.git -q
!pip install ftfy regex tqdm pandas matplotlib scikit-learn opencv-python-headless mediapipe retina-face xgboost openpyxl seaborn -q

# Tải model asset MediaPipe face_landmarker.task nếu còn thiếu
import urllib.request
task_file = os.path.join(BACKEND_DIR, "face_landmarker.task")
if not os.path.exists(task_file):
    print("📥 Đang tải MediaPipe face_landmarker.task...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    try:
        urllib.request.urlretrieve(url, task_file)
        print("✅ Đã tải xong face_landmarker.task!")
    except Exception as e:
        print(f"⚠️ {e}")
else:
    print("✅ Đã có sẵn face_landmarker.task.")

print("🎉 Môi trường đã sẵn sàng 100%!")


## 4️⃣ Nạp Dữ Liệu RAF-DB (Real-world Affective Faces Database)
Vì bộ dữ liệu hình ảnh RAF-DB có dung lượng lớn nên không được đưa lên GitHub. Bạn hãy kết nối Google Drive (hoặc upload file zip) để nạp dữ liệu vào dự án:
- **Cách 1 (Khuyên dùng)**: Đặt file `RAF_DB.zip` hoặc thư mục `RAF_DB` trong Google Drive của bạn (VD: `MyDrive/RAF_DB.zip` hoặc `MyDrive/RAF_DB`).
- **Cách 2**: Upload trực tiếp file `RAF_DB.zip` vào thư mục `/content` của Colab.


In [ ]:
#@title 📦 Nạp Dữ liệu RAF-DB từ Google Drive hoặc File Zip { run: "auto" }
from google.colab import drive
import os
import zipfile
import shutil

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
    try:
        drive.mount("/content/drive")
        print("✅ Đã kết nối Google Drive!")
    except Exception as e:
        print(f"ℹ️ Bỏ qua mount Drive ({e}). Bạn có thể kéo thả file zip trực tiếp vào Colab.")

# Đường dẫn dữ liệu trên Drive (tùy chỉnh nếu bạn để ở thư mục khác)
DRIVE_ZIP_PATH = "/content/drive/MyDrive/RAF_DB.zip" #@param {type:"string"}
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/RAF_DB" #@param {type:"string"}

LOCAL_RAF_DIR = os.path.join(PROJECT_DIR, "RAF_DB")
DATASET_DIR = os.path.join(LOCAL_RAF_DIR, "DATASET")

if not os.path.exists(DATASET_DIR):
    # Kiểm tra các vị trí file zip có thể có
    zip_candidates = [
        DRIVE_ZIP_PATH,
        os.path.join(PROJECT_DIR, "RAF_DB.zip"),
        os.path.join(PROJECT_DIR, "RAF-DB.zip"),
        "/content/RAF_DB.zip",
        "/content/RAF-DB.zip"
    ]
    found_zip = None
    for z in zip_candidates:
        if os.path.exists(z):
            found_zip = z
            break

    if found_zip:
        print(f"📦 Đang giải nén {found_zip} vào {PROJECT_DIR}...")
        !unzip -q "{found_zip}" -d "{PROJECT_DIR}"
        print("✅ Giải nén hoàn tất!")
    elif os.path.exists(DRIVE_FOLDER_PATH):
        print(f"📁 Tìm thấy thư mục RAF_DB trên Drive. Đang tạo liên kết (symlink)...")
        if os.path.exists(LOCAL_RAF_DIR):
            if os.path.islink(LOCAL_RAF_DIR):
                os.remove(LOCAL_RAF_DIR)
            else:
                shutil.rmtree(LOCAL_RAF_DIR)
        os.symlink(DRIVE_FOLDER_PATH, LOCAL_RAF_DIR)
        print("✅ Đã liên kết thư mục RAF_DB từ Google Drive thành công!")
    else:
        print(f"⚠️ Chưa tìm thấy dữ liệu RAF_DB tại: {DATASET_DIR}")
        print("💡 Hãy đảm bảo bạn đã:")
        print("   1. Upload file 'RAF_DB.zip' vào Google Drive (MyDrive/RAF_DB.zip)")
        print("   2. Hoặc kéo thả file 'RAF_DB.zip' vào cây thư mục bên trái của Colab.")

# Kiểm tra số lượng ảnh
if os.path.exists(DATASET_DIR):
    n_train = sum([len(files) for _, _, files in os.walk(os.path.join(DATASET_DIR, "train"))])
    n_test = sum([len(files) for _, _, files in os.walk(os.path.join(DATASET_DIR, "test"))])
    print(f"\n🎉 DỮ LIỆU RAF-DB ĐÃ SẴN SÀNG: {n_train} ảnh train | {n_test} ảnh test")
else:
    print(f"\n❌ Vui lòng kiểm tra lại dữ liệu tại: {DATASET_DIR}")


## 5️⃣ Mẹo Chống Colab Ngắt Kết Nối (Keep-Alive)
Quá trình huấn luyện 100 Epochs có thể kéo dài ~1 - 1.5 tiếng. Để Colab không tự động ngắt kết nối:
- Giữ tab trình duyệt Google Colab luôn mở.
- Nhấn phím `F12` (hoặc `Ctrl + Shift + I` trên Windows), chuyển sang thẻ **Console**, dán đoạn mã sau và nhấn **Enter**:
```javascript
function ClickConnect(){
  console.log("Colab Keep-Alive Heartbeat"); 
  document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(ClickConnect, 60000);
```


---
## 6️⃣ 🚀 THỰC THI LỆNH CHÍNH (Run Complete Pipeline)
Chạy toàn bộ quy trình: Sequential Unfreeze (100 Epochs) $\rightarrow$ Feature Extraction 544D $\rightarrow$ EPSO Selection (289D) $\rightarrow$ Ensemble Head (ResMLP + XGBoost).


In [ ]:
#@title 🚀 Bắt đầu Huấn Luyện Toàn Diện (100 Epochs, Patience 15)
import os
import sys

os.chdir(PROJECT_DIR)
os.environ["PROJECT_ROOT"] = PROJECT_DIR

# Lệnh chính xác theo yêu cầu:
# (Trên Colab/Linux, python thay cho backend\venv\Scripts\python.exe)
!python backend/evaluation/run_complete_pipeline.py --dataset rafdb --epochs 100 --patience 15


---
## 7️⃣ Đánh Giá Kết Quả & Ma Trận Nhầm Lẫn (Confusion Matrix)
Sau khi quá trình huấn luyện hoàn tất, hiển thị ma trận nhầm lẫn và kết quả phân loại:


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

cm_paths = [
    os.path.join(BACKEND_DIR, "confusion_matrix.png"),
    os.path.join(BACKEND_DIR, "confusion_matrix_rafdb.png"),
    os.path.join(BACKEND_DIR, "reports", "confusion_matrix.png")
]

found_cm = None
for p in cm_paths:
    if os.path.exists(p):
        found_cm = p
        break

if found_cm:
    print(f"📈 Ma trận Nhầm lẫn đã tạo: {found_cm}")
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(found_cm))
    plt.axis("off")
    plt.title("Confusion Matrix - RAF-DB Evaluation (EduFocus AI)", fontsize=14, fontweight="bold")
    plt.show()
else:
    print("ℹ️ Chạy benchmark để kiểm tra lại kết quả:")
    !python backend/evaluation/benchmark.py


---
## 8️⃣ Lưu Trọng Số (Weights) Vào Google Drive & Tải Về Máy
Tự động sao lưu toàn bộ trọng số tốt nhất vào thư mục `MyDrive/edufocus_weights_rafdb/` trên Google Drive để không bị mất khi đóng phiên Colab, đồng thời tạo file zip tải về máy:


In [ ]:
#@title 💾 Sao Lưu Trọng Số Vào Google Drive & Tải File Zip
import zipfile
import os
import shutil
from google.colab import files

# 1. Danh sách các file trọng số quan trọng
weights_to_save = [
    "models/clip_finetuned_best.pth",
    "models/facial_cross_attn_best.pth",
    "models/visual_proj_best.pth",
    "emotion_resmlp_weights.pth",
    "emotion_mlp_epso_weights.pth",
    "emotion_mlp_best_acc_weights_rafdb.pth",
    "emotion_mlp_epso_weights_rafdb.pth",
    "xgboost_model.json",
    "epso_mask_real.npy",
    "ensemble_weights.json"
]

# 2. Tự động sao lưu thẳng vào Google Drive nếu đã mount
drive_backup_dir = "/content/drive/MyDrive/edufocus_weights_rafdb"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(drive_backup_dir, exist_ok=True)
    for rel_path in weights_to_save:
        src = os.path.join(BACKEND_DIR, rel_path)
        if os.path.exists(src):
            dst = os.path.join(drive_backup_dir, rel_path)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            print(f"  💾 Đã lưu vào Drive: {rel_path}")
    print(f"\n✅ ĐÃ SAO LƯU TOÀN BỘ VÀO GOOGLE DRIVE: {drive_backup_dir}")

# 3. Tạo gói nén zip để tải về máy
zip_name = "edufocus_rafdb_best_weights.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for rel_path in weights_to_save:
        full_path = os.path.join(BACKEND_DIR, rel_path)
        if os.path.exists(full_path):
            zipf.write(full_path, arcname=rel_path)
            print(f"  + Đã nén: {rel_path} ({os.path.getsize(full_path)/1e6:.2f} MB)")
        else:
            print(f"  - Chưa có: {rel_path}")

print(f"\n✅ Đã tạo gói nén: {zip_name} ({os.path.getsize(zip_name)/1e6:.2f} MB)")
print("📥 Đang gửi lệnh tải file về máy tính...")
try:
    files.download(zip_name)
except Exception as e:
    print(f"ℹ️ File zip lưu tại {os.path.abspath(zip_name)}. Bạn có thể click chuột phải tải về từ thanh Files của Colab.")
